# Text-to-SQL QLoRA training (Colab)

Thin wrapper around `src/train.py` — all real logic lives in the repo, not here, so Colab/Kaggle stay identical. Runs 3 ablation ranks (r=8/16/32) then the final full-budget run with the winning rank.

**Checkpoints save to Google Drive** so a ~90-min idle disconnect loses at most `save_steps` worth of progress — re-running the same cell resumes automatically.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_ROOT = '/content/drive/MyDrive/text-to-sql-qlora/checkpoints'
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
# Export to the OS environment, not just the Python namespace — "!" shell
# cells below resolve $CHECKPOINT_ROOT from os.environ, not from this variable.
os.environ['CHECKPOINT_ROOT'] = CHECKPOINT_ROOT

Mounted at /content/drive


In [2]:
import os
REPO_DIR = '/content/text-to-sql-qlora'  # explicit dir name — avoids the leading '-' in the GitHub repo name, which breaks %cd's flag parser
os.environ['REPO_DIR'] = REPO_DIR

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git clone https://github.com/Rick-Roy-JC/-text-to-sql-qlora.git "$REPO_DIR"
else:
    print(f"{REPO_DIR} already cloned — pulling latest instead.")
    !git -C "$REPO_DIR" pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt

Cloning into '/content/text-to-sql-qlora'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 71 (delta 22), reused 63 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 18.53 MiB | 32.33 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/text-to-sql-qlora
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.2 MB/s eta 0:00:00


## Ablation runs: r=8, r=16, r=32 (2,500-example subset, 2 epochs each, ~25-30 min/run on T4)

In [ ]:
%cd {REPO_DIR}
!python src/train.py --regime ablation --lora-rank 8 --output-root "$CHECKPOINT_ROOT"

In [3]:
%cd {REPO_DIR}
!python src/train.py --regime ablation --lora-rank 16 --output-root "$CHECKPOINT_ROOT"

/content/text-to-sql-qlora
config.json: 100% 967/967 [00:00<00:00, 3.98MB/s]
tokenizer_config.json: 100% 3.44k/3.44k [00:00<00:00, 10.9MB/s]
tokenizer.json: 100% 1.94M/1.94M [00:00<00:00, 78.3MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 654kB/s]
added_tokens.json: 100% 306/306 [00:00<00:00, 1.80MB/s]
special_tokens_map.json: 100% 599/599 [00:00<00:00, 2.78MB/s]
model.safetensors.index.json: 100% 16.5k/16.5k [00:00<00:00, 48.5MB/s]
Fetching 2 files: 100% 2/2 [02:23<00:00, 71.89s/it] 
Download complete: 100% 7.64G/7.64G [02:23<00:00, 53.1MB/s]
Loading weights:   1% 2/195 [00:00<00:49,  3.88it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 195/195 [00:22<00:00,  8.72it/s]
generation_config.json: 100% 181/181 [00:00<00:00, 742kB/s]
trainable params: 8,912,89

In [ ]:
%cd {REPO_DIR}
!python src/train.py --regime ablation --lora-rank 32 --output-root "$CHECKPOINT_ROOT"

## Compare ablation results

Inspect `results/train_metrics_ablation_r*.json` (eval loss on val split) before picking the winning rank for the final run. Full exact-match/execution-accuracy comparison happens in Phase 4 (`src/eval.py`), not here.

In [ ]:
%cd {REPO_DIR}
import json, glob
for path in sorted(glob.glob('results/train_metrics_ablation_r*.json')):
    print(path, json.load(open(path)))

## Final run: winning rank, full ~6,294-example train split, 3 epochs

Set `WINNING_RANK` based on the ablation comparison above.

In [ ]:
WINNING_RANK = 16  # update after inspecting ablation results
import os
os.environ['WINNING_RANK'] = str(WINNING_RANK)
%cd {REPO_DIR}
!python src/train.py --regime final --lora-rank $WINNING_RANK --output-root "$CHECKPOINT_ROOT"